In [2]:
import chardet,pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.impute import SimpleImputer

In [3]:
file_path='/content/global_student_migration.csv%3FX-Goog-Algorithm=GOOG4-RSA-SHA256&X-Goog-Credential=gcp-kaggle-com@kaggle-161607.iam.gserviceaccount.com%2F20251021%2Fauto%2Fstorage%2Fgoog4_request&X-Goog-Date=20251021T045248Z&X-Goog-Expires=259.csv'

In [4]:
with open(file_path, 'rb') as f:
    enc = chardet.detect(f.read(100000))['encoding']

In [5]:
df = pd.read_csv(file_path,encoding=enc, sep=',', skipinitialspace=True, na_values=['N/A','NA','none','None'], dtype=str)

In [6]:
print("Raw Data Shape:",df.shape)
print(df.head(10))

Raw Data Shape: (5000, 21)
  student_id origin_country destination_country destination_city  \
0     S00001        Finland              Russia           Moscow   
1     S00002             UK             Germany           Aachen   
2     S00003        Ireland              Canada        Vancouver   
3     S00004            UAE                  UK       Birmingham   
4     S00005   South Africa             Germany        Stuttgart   
5     S00006            UAE        South Africa         Pretoria   
6     S00007            UAE                  UK        Edinburgh   
7     S00008            UAE                  UK        Cambridge   
8     S00009          India              Russia           Moscow   
9     S00010        Germany               India           Mumbai   

                     university_name              course_name  \
0  Lomonosov Moscow State University         Computer Science   
1                        RWTH Aachen        Civil Engineering   
2     University of British C

In [7]:
df.drop_duplicates(inplace=True)

In [8]:
df.columns = df.columns.str.strip().str.lower()

In [9]:
print("Missing Values Summary:")
print(df.isnull().sum())

Missing Values Summary:
student_id                      0
origin_country                  0
destination_country             0
destination_city                0
university_name                 0
course_name                     0
field_of_study                  0
year_of_enrollment              0
scholarship_received            0
enrollment_reason               0
graduation_year                 0
placement_status                0
placement_country            2491
placement_company            2491
starting_salary_usd             0
gpa_or_score                    0
visa_status                     0
post_graduation_visa            0
language_proficiency_test     982
test_score                      0
world_university_rank        2897
dtype: int64


In [10]:
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

In [11]:
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

In [12]:
num_cols = [
    'year_of_enrollment','graduation_year',
    'starting_salary_usd','gpa_or_score',
    'test_score','world_university_rank'
]

In [13]:
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [14]:
cat_cols = [c for c in df.columns if c not in num_cols]

In [15]:
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')
df[num_cols] = num_imputer.fit_transform(df[num_cols])
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

At this step we have processed all missing values


In [16]:
print(df[num_cols].dtypes)
print(df.isnull().sum().sum())

year_of_enrollment       float64
graduation_year          float64
starting_salary_usd      float64
gpa_or_score             float64
test_score               float64
world_university_rank    float64
dtype: object
0


In [17]:
df['study_duration'] = df['graduation_year'] - df['year_of_enrollment']

In [18]:
df['stem_flag'] = df['field_of_study'].apply(
    lambda x: 1 if str(x).lower() in ['engineering','computer science','mathematics','technology','science'] else 0
)

In [19]:
df['placed_flag'] = df['placement_status'].apply(lambda x: 1 if str(x).lower() == 'placed' else 0)

In [20]:
df['stay_in_canada'] = df['placement_country'].apply(lambda x: 1 if str(x).strip().lower() == 'canada' else 0)

In [21]:
df['scholarship_flag'] = df['scholarship_received'].apply(lambda x: 1 if str(x).lower().startswith('y') else 0)


In [22]:
if 'starting_salary_usd' in df.columns:
    q1, q3 = df['starting_salary_usd'].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df.loc[df['starting_salary_usd'] < lower, 'starting_salary_usd'] = np.nan
    df.loc[df['starting_salary_usd'] > upper, 'starting_salary_usd'] = np.nan
    df['starting_salary_usd'].fillna(df['starting_salary_usd'].median(), inplace=True)


/tmp/ipython-input-1684376812.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['starting_salary_usd'].fillna(df['starting_salary_usd'].median(), inplace=True)


In [23]:
label_cols = [
    'origin_country', 'destination_country', 'destination_city',
    'university_name', 'course_name', 'field_of_study',
    'enrollment_reason', 'placement_company',
    'visa_status', 'post_graduation_visa',
    'language_proficiency_test'
]

le = LabelEncoder()
for col in label_cols:
    df[col] = le.fit_transform(df[col].astype(str))

In [24]:
scaler = StandardScaler()
to_scale = ['starting_salary_usd', 'gpa_or_score', 'world_university_rank', 'test_score']
df[to_scale] = scaler.fit_transform(df[to_scale])

In [25]:
target_col = 'stay_in_canada'
X = df.drop(target_col, axis=1)
y = df[target_col].astype(int)

In [26]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, stratify=y, test_size=0.3, random_state=42
)

In [27]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, stratify=y_temp, test_size=0.5, random_state=42
)

In [29]:
train_df = pd.concat([X_train, y_train], axis=1)
val_df   = pd.concat([X_val, y_val], axis=1)
test_df  = pd.concat([X_test, y_test], axis=1)

In [30]:
train_df.to_csv('train_preprocessed.csv', index=False)
val_df.to_csv('val_preprocessed.csv', index=False)
test_df.to_csv('test_preprocessed.csv', index=False)

print("Data preprocessing pipeline executed successfully")
print(f"Train:{train_df.shape}, Validation:{val_df.shape}, Test:{test_df.shape}")

Data preprocessing pipeline executed successfully
Train:(3500, 26), Validation:(750, 26), Test:(750, 26)


In [32]:
print("\n Check Key Stats:")
print(train_df[['starting_salary_usd', 'gpa_or_score', 'world_university_rank', 'stay_in_canada']].describe())


 Check Key Stats:
       starting_salary_usd  gpa_or_score  world_university_rank  \
count          3500.000000   3500.000000            3500.000000   
mean              0.000834     -0.002004              -0.005427   
std               1.001699      1.003441               0.992288   
min              -0.878504     -1.734308              -0.893253   
25%              -0.878504     -0.875073              -0.285495   
50%              -0.277941     -0.039060              -0.285495   
75%               0.858621      0.866620              -0.285495   
max               2.077261      1.749078               4.834405   

       stay_in_canada  
count     3500.000000  
mean         0.050571  
std          0.219152  
min          0.000000  
25%          0.000000  
50%          0.000000  
75%          0.000000  
max          1.000000  
